In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Install and Imports

In [ ]:
%%capture
!pip install sentencepiece
!pip install transformers
!pip install nltk
!pip install matplotlib

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.optimization import AdamW
from tqdm import tqdm

C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model

In [ ]:
# Define the dataset class
class KeyTextDataset(Dataset):
    def __init__(self, keys, texts, tokenizer):
        self.keys = keys
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        text = self.texts[idx]
        key_encoding = self.tokenizer(
            key,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )

        text_encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = key_encoding['input_ids'].squeeze()
        attention_mask = key_encoding['attention_mask'].squeeze()

        # print(text_encoding)

        labels = text_encoding['input_ids'].squeeze()
        labels[labels == 0] = -100
        labels_attention_mask = text_encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_attention_mask':labels_attention_mask,
            'text': text
        }

# Function to train the model
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to validate the model
def validate_model(model, dataloader, device):
    model.eval()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to save the trained model and tokenizer
def save_model(model, tokenizer, output_dir):
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to '{output_dir}'")

# Function to load the saved model and tokenizer
def load_model(output_dir):
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print(f"Model and tokenizer loaded from '{output_dir}'")
    return model, tokenizer

In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model_dir = "csebuetnlp/mT5_multilingual_XLSum"
# model_dir = "csebuetnlp/banglat5"
# model_dir = "./Model/ModelV25"
model_dir = "./Model/bnT5ModelV25"
# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(device)

## Dataset Load

In [ ]:
import pandas as pd

df = pd.read_csv('./Data/test/testData1M.csv')
df = df.head(10000)
df

,keywords,text
0,জলাশয়ের বনাঞ্চল না কখনো পরিবেশ এমন করব স্থান,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট..."
1,একাধিক একটি দমনে টহল বাহিনীর মেঘনায়,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...
2,প্রশ্ন তখন নিয়েই হতো প্রশ্ন এখন নিয়ে,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ..."
3,টাকা নেয় থেকে লোক কালিয়াকৈরের পকেটে সদস্যঅপহরণ...,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...
4,বেশি হাজারের জেলে ৪১ কর্মহীন এতে,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...
...,...,...
9995,শিক্ষার্থী ও আলোচনা উত্তর নমুনা করব অধ্যায়—১ ...,"বহুনির্বাচনি প্রশ্নোত্তরপ্রিয় শিক্ষার্থী, আজ ..."
9996,ভিয়েনায় জাতিসংঘ অস্ট্রিয়ার দাবিতে শান্তিপূর্ণ ...,মিয়ানমারের রাখাইন রাজ্যে রোহিঙ্গা মুসলমানদের ব...
9997,আকিদা মানুষ অবিলম্বে গণভবনে প্রধানমন্ত্রী নেমে...,"অবিলম্বে এটি অপসারণের দাবি জানিয়ে তিনি বলেন, অ..."
9998,দেড় ১৩২ শ সোজা করা করার প্রস্তাব এই প্রতিরক্ষ...,এই প্রকল্পে সাড়ে ১০ কিলোমিটার স্থায়ী প্রতিরক্ষ...


In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
# Load your dataset
keys = df['keywords'].tolist()  # List of keys
texts = df['text'].tolist()  # List of corresponding texts

In [ ]:
type(texts)

list

## Prediction

In [ ]:
# loading_model_dir = "./Model/ModelV26"
loading_model_dir = "./Model/bnT5ModelV26"
loaded_model, loaded_tokenizer = load_model(loading_model_dir)
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)

In [ ]:
# Function to generate text given a key
def generate_text(key):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)

    with torch.no_grad():
      outputs = loaded_model.generate(
          input_ids=input_ids,
          max_length =64,
          # max_new_tokens = 64,
          # num_beams =2,
          # num_beams = 1, # For Greedy
          # early_stopping =True,
          num_return_sequences = 1,
          temperature = 0.3,
          # top_k= 50,
          top_p= 0.95,
          do_sample=True,
          # do_sample=False, # For Greedy
          repetition_penalty= 2.5,
          length_penalty= 1.0)

    # print(outputs)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]
    generated_text = preds[0]

    return generated_text

def predict(key):
  return generate_text(key)

In [ ]:
key = "কেমন ডাটাসেট সময় বানাতে"
predict(key)

'ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'

In [ ]:
keywords = "নির্বাহের জীবিকা বৃদ্ধার সম্বল জমিটাই"
predict(keywords)

'ছোটবেলার স্মৃতি মনে হয় না, তাঁকে দেখা হয়নি।'

In [ ]:
genDf = pd.DataFrame(columns = ["keywords", "text", "generatedText"])

rows = df.shape[0]

for i in range(rows):
  keywords = df['keywords'][i]
  generatedText = predict(keywords)

  genDf.loc[i] = [df['keywords'][i], df['text'][i], generatedText]
#   print(i, '  ', df['keywords'][i], '\n', df['text'][i], '\n', generatedText)

In [ ]:
genDf.to_csv('./Data/test/toppAndMaxLength/predictedData10.csv', index=False)

In [ ]:
import pandas as pd

url = './Data/test/toppAndMaxLength/predictedData'

df1 = pd.read_csv(url + "1.csv")
df2 = pd.read_csv(url + "2.csv")
df3 = pd.read_csv(url + "3.csv")
df4 = pd.read_csv(url + "4.csv")
df5 = pd.read_csv(url + "5.csv")
df6 = pd.read_csv(url + "6.csv")
df7 = pd.read_csv(url + "7.csv")
df8 = pd.read_csv(url + "8.csv")
df9 = pd.read_csv(url + "9.csv")
df10 = pd.read_csv(url + "10.csv")

frames = [df1, df2, df3, df4, df5, df6]
genDfCombined = pd.concat(frames)
genDfCombined.to_csv(url + ".csv", index=False)

## Get Score

### BLEU

In [ ]:
import nltk

def bleuCorpass(df, columnName1, columnName2):
    ble1score = 0
    ble2score = 0
    ble3score = 0
    ble4score = 0
    lenS = df.shape[0]
    for i in range(lenS):
        hypothesis = df[columnName2][i].split()
        reference = df[columnName1][i].split()
        references = [reference]
        list_of_references = [references]
        list_of_hypotheses = [hypothesis]
        ble1score = ble1score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(1, 0, 0, 0))
        ble2score = ble2score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.5, 0.5, 0, 0))
        ble3score = ble3score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.33, 0.33, 0.33, 0))
        ble4score = ble4score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

    bscore = {'Bleu 1': ble1score/lenS,
              'Bleu 2': ble2score/lenS,
              'Bleu 3': ble3score/lenS,
              'Bleu 4': ble4score/lenS}

    return bscore

### Meteor WIL WER

In [ ]:
from collections import Counter

class MeteorScore:
    def __init__(self, alpha=0.5, beta=0.5, gamma=0.5):
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def preprocess_sentence(self, sentence):
        words = sentence.split()
        return words

    def ngram_count(self, sentence, n):
        words = self.preprocess_sentence(sentence)
        ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
        return Counter(ngrams)

    def compute_precision(self, hypothesis, reference, n):
        hyp_counts = self.ngram_count(hypothesis, n)
        ref_counts = self.ngram_count(reference, n)
        overlap = sum((hyp_counts & ref_counts).values())
        precision = overlap / sum(hyp_counts.values()) if sum(hyp_counts.values()) > 0 else 0
        return precision

    def compute_recall(self, hypothesis, reference, n):
        hyp_counts = self.ngram_count(hypothesis, n)
        ref_counts = self.ngram_count(reference, n)
        overlap = sum((hyp_counts & ref_counts).values())
        recall = overlap / sum(ref_counts.values()) if sum(ref_counts.values()) > 0 else 0
        return recall

    def meteor_score(self, hypothesis, reference):
        precision = self.alpha * self.compute_precision(hypothesis, reference, 1) + (1-self.alpha) * self.compute_precision(hypothesis, reference, 2)
        recall = self.beta * self.compute_recall(hypothesis, reference, 1) + (1-self.beta) * self.compute_recall(hypothesis, reference, 2)
        fmean = (1-self.gamma) * precision + self.gamma * recall if (precision != 0 and recall != 0) else 0
        return fmean

In [ ]:
%%capture
!pip install jiwer

In [ ]:
import jiwer

meteor = MeteorScore(alpha=0.5, beta=0.5, gamma=0.5)

def wilWerMeteorScore(df, columnName1, columnName2):
    df = df.filter([columnName1, columnName2], axis=1)

    lenS = df.shape[0]
    meteor_score = 0
    wer_score = 0
    wil_score = 0

    for i in range(lenS):
      meteor_result = meteor.meteor_score(df[columnName2][i], df[columnName1][i])
      wer_result = jiwer.wer(df[columnName1][i], df[columnName2][i])
      wil_result = jiwer.wil(df[columnName1][i], df[columnName2][i])

      meteor_score = meteor_score + meteor_result
      wer_score = wer_score + wer_result
      wil_score = wil_score + wil_result

    wilWerMeteor_score = {'METEOR': meteor_score/lenS,
                    'WER': wer_score/lenS,
                    'WIL': wil_score/lenS}

    return wilWerMeteor_score

### Scores

In [ ]:
import pandas as pd
# df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')
df =  pd.read_csv('./Data/test/predictedDataGreedy1k.csv')

In [ ]:
df

,keywords,text,generatedText
0,জলাশয়ের বনাঞ্চল না কখনো পরিবেশ এমন করব স্থান,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট...","জলাশয়ের এমন একটি বনাঞ্চল, যেখানে পরিবেশ কখনো ..."
1,একাধিক একটি দমনে টহল বাহিনীর মেঘনায়,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...,মেঘনায় হামলা দমনে আইনশৃঙ্খলার একাধিক বাহিনীর ...
2,প্রশ্ন তখন নিয়েই হতো প্রশ্ন এখন নিয়ে,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ...","এখন প্রশ্ন হতো, তবে কীভাবে সেটা দেখা যায়?প্রশ..."
3,টাকা নেয় থেকে লোক কালিয়াকৈরের পকেটে সদস্যঅপহরণ...,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...,গাজীপুরের কালিয়াকৈরের চরপাড়া এলাকায় বাসে কর...
4,বেশি হাজারের জেলে ৪১ কর্মহীন এতে,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...,এতে ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়েছে।
...,...,...,...
995,নির্মিত দশকে নামে বিল্ডিং জাহাজ বাড়িটি পরিচিত,তবে আশির দশকে নির্মিত ছয়তলা এই বাড়িটি ‘জাহাজ ব...,বাড়িটি ১৯৭১-এর দশকে জাহাজ বিল্ডিং নামে পরিচিত...
996,এস্টেট করেন আহ্বান মঞ্চে ইনভেস্টর হোসেন তাঁকে,তাঁকে মঞ্চে আহ্বান করেন রিয়েল এস্টেট ইনভেস্টর ...,মঞ্চে তাঁকে ফুল দিয়ে আহ্বান করেছেন ইনভেস্টর এ...
997,রোহিঙ্গারা ভারতের করতে জন্য এবং কাজ স্বরাষ্ট্র...,এর আগে ভারতের রাজ্যগুলোর প্রতি দেওয়া এক পরামর্...,এর আগে রোহিঙ্গারা তাদের প্রতি আস্থা দেওয়া এবং...
998,কেজরিওয়াল এ বাস্তবায়ন ক্ষেত্রে সুবিধা তাঁর নাগ...,এ ক্ষেত্রে অরবিন্দ কেজরিওয়াল কিছু নাগরিক সুবিধ...,এ ক্ষেত্রে তাঁর শিক্ষা সুবিধা কিছুটা কম।ভারতের...


In [ ]:
bleuCorpass(df, 'text', 'generatedText')

C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-g

{'Bleu 1': 0.37976768366058394,
 'Bleu 2': 0.170029638776799,
 'Bleu 3': 0.06063591163929269,
 'Bleu 4': 0.02298000216690158}

In [ ]:
wilWerMeteorScore(df, 'text', 'generatedText')

{'METEOR': 0.2704952388257781,
 'WER': 0.8742024217412636,
 'WIL': 0.9030891387938094}

### Bangla BERTScore

In [ ]:
import pandas as pd
# df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')
df

,keywords,text,generatedText
0,জলাশয়ের বনাঞ্চল না কখনো পরিবেশ এমন করব স্থান,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট...","জলাশয়ের এমন একটি বনাঞ্চল, যেখানে পরিবেশ কখনো ..."
1,একাধিক একটি দমনে টহল বাহিনীর মেঘনায়,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...,মেঘনায় হামলা দমনে আইনশৃঙ্খলার একাধিক বাহিনীর ...
2,প্রশ্ন তখন নিয়েই হতো প্রশ্ন এখন নিয়ে,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ...","এখন প্রশ্ন হতো, তবে কীভাবে সেটা দেখা যায়?প্রশ..."
3,টাকা নেয় থেকে লোক কালিয়াকৈরের পকেটে সদস্যঅপহরণ...,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...,গাজীপুরের কালিয়াকৈরের চরপাড়া এলাকায় বাসে কর...
4,বেশি হাজারের জেলে ৪১ কর্মহীন এতে,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...,এতে ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়েছে।
...,...,...,...
995,নির্মিত দশকে নামে বিল্ডিং জাহাজ বাড়িটি পরিচিত,তবে আশির দশকে নির্মিত ছয়তলা এই বাড়িটি ‘জাহাজ ব...,বাড়িটি ১৯৭১-এর দশকে জাহাজ বিল্ডিং নামে পরিচিত...
996,এস্টেট করেন আহ্বান মঞ্চে ইনভেস্টর হোসেন তাঁকে,তাঁকে মঞ্চে আহ্বান করেন রিয়েল এস্টেট ইনভেস্টর ...,মঞ্চে তাঁকে ফুল দিয়ে আহ্বান করেছেন ইনভেস্টর এ...
997,রোহিঙ্গারা ভারতের করতে জন্য এবং কাজ স্বরাষ্ট্র...,এর আগে ভারতের রাজ্যগুলোর প্রতি দেওয়া এক পরামর্...,এর আগে রোহিঙ্গারা তাদের প্রতি আস্থা দেওয়া এবং...
998,কেজরিওয়াল এ বাস্তবায়ন ক্ষেত্রে সুবিধা তাঁর নাগ...,এ ক্ষেত্রে অরবিন্দ কেজরিওয়াল কিছু নাগরিক সুবিধ...,এ ক্ষেত্রে তাঁর শিক্ষা সুবিধা কিছুটা কম।ভারতের...


In [ ]:
cands = df["generatedText"].values.tolist()
refs = df["text"].values.tolist()

In [ ]:
print(type(cands), ' ',len(cands), '\n', type(refs), ' ',len(refs))

<class 'list'>   1000 
 <class 'list'>   1000


In [ ]:
!git clone https://github.com/csebuetnlp/banglaparaphrase.git
%cd ./banglaparaphrase/BERTScore/utils

C:\Users\USER\DeepLearning\bnKey2Text\banglaparaphrase\BERTScore\utils


fatal: destination path 'banglaparaphrase' already exists and is not an empty directory.


In [ ]:
import os

current_path = os.getcwd()
print("Current Path:", current_path)

Current Path: C:\Users\USER\DeepLearning\bnKey2Text\banglaparaphrase\BERTScore\utils


In [ ]:
%%capture
!pip install transformers
!pip install git+https://github.com/csebuetnlp/normalizer.git
!pip install jsonlines

  Cloning https://github.com/csebuetnlp/normalizer.git to c:\users\user\appdata\local\temp\pip-req-build-6crxufj1
  Resolved https://github.com/csebuetnlp/normalizer.git to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer.git 'C:\Users\USER\AppData\Local\Temp\pip-req-build-6crxufj1'


In [ ]:
from score import score
P, R, F1 = score(cands, refs, lang='bn', verbose=True)

F1_mean= F1.mean()

C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


calculating scores...
computing bert embedding.


100%|██████████| 32/32 [00:14<00:00,  2.20it/s]


computing greedy matching.


100%|██████████| 16/16 [00:00<00:00, 21.89it/s]


done in 15.46 seconds, 64.70 sentences/sec


In [ ]:
print(F1_mean)

tensor(0.8903)


### Multilingual ROUGE

In [ ]:
import os

new_path = r'C:\Users\USER\DeepLearning\bnKey2Text'

os.chdir(new_path)

current_path = os.getcwd()
print("Current Path:", current_path)

Current Path: C:\Users\USER\DeepLearning\bnKey2Text


In [ ]:
import pandas as pd
# df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')
df

,keywords,text,generatedText
0,জলাশয়ের বনাঞ্চল না কখনো পরিবেশ এমন করব স্থান,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট...","জলাশয়ের এমন একটি বনাঞ্চল, যেখানে পরিবেশ কখনো ..."
1,একাধিক একটি দমনে টহল বাহিনীর মেঘনায়,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...,মেঘনায় হামলা দমনে আইনশৃঙ্খলার একাধিক বাহিনীর ...
2,প্রশ্ন তখন নিয়েই হতো প্রশ্ন এখন নিয়ে,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ...","এখন প্রশ্ন হতো, তবে কীভাবে সেটা দেখা যায়?প্রশ..."
3,টাকা নেয় থেকে লোক কালিয়াকৈরের পকেটে সদস্যঅপহরণ...,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...,গাজীপুরের কালিয়াকৈরের চরপাড়া এলাকায় বাসে কর...
4,বেশি হাজারের জেলে ৪১ কর্মহীন এতে,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...,এতে ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়েছে।
...,...,...,...
995,নির্মিত দশকে নামে বিল্ডিং জাহাজ বাড়িটি পরিচিত,তবে আশির দশকে নির্মিত ছয়তলা এই বাড়িটি ‘জাহাজ ব...,বাড়িটি ১৯৭১-এর দশকে জাহাজ বিল্ডিং নামে পরিচিত...
996,এস্টেট করেন আহ্বান মঞ্চে ইনভেস্টর হোসেন তাঁকে,তাঁকে মঞ্চে আহ্বান করেন রিয়েল এস্টেট ইনভেস্টর ...,মঞ্চে তাঁকে ফুল দিয়ে আহ্বান করেছেন ইনভেস্টর এ...
997,রোহিঙ্গারা ভারতের করতে জন্য এবং কাজ স্বরাষ্ট্র...,এর আগে ভারতের রাজ্যগুলোর প্রতি দেওয়া এক পরামর্...,এর আগে রোহিঙ্গারা তাদের প্রতি আস্থা দেওয়া এবং...
998,কেজরিওয়াল এ বাস্তবায়ন ক্ষেত্রে সুবিধা তাঁর নাগ...,এ ক্ষেত্রে অরবিন্দ কেজরিওয়াল কিছু নাগরিক সুবিধ...,এ ক্ষেত্রে তাঁর শিক্ষা সুবিধা কিছুটা কম।ভারতের...


In [ ]:
%%capture
!git clone https://github.com/csebuetnlp/xl-sum.git
%cd ./xl-sum/multilingual_rouge_scoring
!pip3 install -r requirements.txt
!pip3 install --upgrade ./

In [ ]:
from rouge_score import rouge_scorer

rougescorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True, lang="bengali")

def rougeScore2(test_df, columnName1, columnName2):

    newDf = test_df.filter([columnName1, columnName2], axis=1)

    newDf =  newDf.reset_index()

    lenS = newDf.shape[0]

    rscore1 = 0
    rscoreL = 0

    for i in range(lenS):
        score = rougescorer.score(newDf[columnName1][i], newDf[columnName2][i])

        rscore1 = rscore1 + score['rouge1'].fmeasure
        rscoreL = rscoreL + score['rougeL'].fmeasure

    print(rscore1/lenS)
    print(rscoreL/lenS)

    return rscoreL/lenS

In [ ]:
rougeScore2(df, 'text', 'generatedText')

0.505393169249115
0.3706899613562351


0.3706899613562351